# macOS Screen Time — data exploration

Reads app usage directly from `knowledgeC.db`.

**Requires Full Disk Access** for your terminal:
System Settings → Privacy & Security → Full Disk Access

In [1]:
import sys
sys.path.insert(0, "..")

from pipeline.extractors.screentime import _query_db, _to_df

In [2]:
rows = _query_db()
print(f"{len(rows)} raw records")
rows[:3]

690 raw records


[('com.google.Chrome', 8, 1774828885, 3600),
 ('com.microsoft.VSCode', 33, 1774828852, 3600),
 ('com.google.Chrome', 9, 1774828843, 3600)]

In [3]:
df = _to_df(rows)
print(df.shape)
df.head(10)

(36, 3)


date,category,value
date,str,f64
2026-03-08,"""com.microsoft.VSCode""",63.3
2026-03-08,"""com.apple.Safari""",0.1
2026-03-08,"""com.google.Chrome""",146.9
2026-03-08,"""com.apple.finder""",0.4
2026-03-08,"""com.apple.Console""",0.1
2026-03-08,"""com.anthropic.claudefordesktop""",0.6
2026-03-09,"""com.google.Chrome""",1.4
2026-03-09,"""com.anthropic.claudefordesktop""",0.0
2026-03-09,"""com.microsoft.VSCode""",0.0


In [ ]:
# Date range
print("From:", df["date"].min())
print("To:  ", df["date"].max())
print("Days:", df["date"].n_unique())

In [ ]:
# Total daily screen time (all apps)
daily = (
    df.group_by("date")
    .agg(pl.col("value").sum().alias("total_minutes"))
    .sort("date")
)
print("Avg daily screen time (min):", daily["total_minutes"].mean().round(1))
daily.tail(14)

In [ ]:
import polars as pl

# Top apps by total usage
(
    df.group_by("category")
    .agg(pl.col("value").sum().alias("total_minutes"))
    .sort("total_minutes", descending=True)
    .head(20)
)